# MAE Analysis

This notebook evaluates the embeddings learned by the MAE tutorial and helps interpret the resulting artifacts. You can point it at either the CIFAR-10 or MNIST MAE config from the selector cell below.


## Learning Goals

By the end of this notebook, you should understand:

- why reconstruction loss alone is not enough to judge a representation
- what linear probing and kNN evaluation are actually measuring
- how to inspect projection and nearest-neighbor artifacts
- how to compare two MAE runs in a principled way

## Why Evaluate A Reconstruction Model With Classification-Like Metrics?

MAE is trained without labels, but in tutorial settings we still need a simple answer to: **did the encoder learn something useful?**

This repository uses three complementary views:

- **linear probe**: can a shallow classifier read class information from the embeddings?
- **kNN**: do semantically similar images cluster nearby in representation space?
- **qualitative inspection**: do projection plots and nearest-neighbor grids look coherent?

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml")

REPO_ROOT = find_repo_root(Path.cwd())

os.chdir(REPO_ROOT)
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from fomocid.utils import format_metrics_markdown, load_config, load_json, open_saved_image

AVAILABLE_MAE_CONFIGS = {
    "cifar10": Path("configs/cifar10_mae.yaml"),
    "mnist": Path("configs/mnist_mae.yaml"),
}
SELECTED_DATASET = "mnist" 
if SELECTED_DATASET not in AVAILABLE_MAE_CONFIGS:
    raise ValueError(f"Unsupported dataset selection: {SELECTED_DATASET}")

CONFIG_PATH = AVAILABLE_MAE_CONFIGS[SELECTED_DATASET]
config = load_config(CONFIG_PATH)
OUTPUT_ROOT = Path("outputs")
print("Repo root detected. Running commands relative to the repository root.")
print(f"Selected dataset: {SELECTED_DATASET}")
print(f"Config path: {CONFIG_PATH}")


## Locate A Checkpoint

The next cell finds the newest `last.ckpt` for the selected MAE dataset under `outputs/`. If you want a specific run, set `checkpoint_path` manually instead.


In [ ]:
run_prefix = f"{config['dataset']['name']}_{config['ssl']['method']}_"
candidates = sorted(OUTPUT_ROOT.glob(f"{run_prefix}*/checkpoints/last.ckpt"), key=lambda path: path.stat().st_mtime)
checkpoint_path = candidates[-1] if candidates else None
checkpoint_path.relative_to(REPO_ROOT) if checkpoint_path is not None and checkpoint_path.is_absolute() else checkpoint_path


## Run The Evaluation Script

Canonical command:

```bash
python tutorials/evaluate_representations.py --config configs/<selected-mae-config>.yaml --checkpoint-path outputs/<run>/checkpoints/last.ckpt --output-dir outputs
```

If `checkpoint_path` is `None`, go back to the pretraining notebook first.


In [ ]:
if checkpoint_path is None:
    print("No checkpoint found yet. Run the pretraining notebook first.")
else:
    _ = subprocess.run(
        [
            sys.executable,
            "tutorials/evaluate_representations.py",
            "--config",
            str(CONFIG_PATH),
            "--checkpoint-path",
            str(checkpoint_path),
            "--output-dir",
            str(OUTPUT_ROOT),
        ],
        check=True,
        cwd=str(REPO_ROOT),
    )
    print("Evaluation finished. Artifacts were written under outputs/.")


In [ ]:
eval_dir = checkpoint_path.parent.parent / "evaluation" if checkpoint_path is not None else None
eval_dir.relative_to(REPO_ROOT) if eval_dir is not None and eval_dir.is_absolute() else eval_dir


## Read The Metrics

The metrics file is deliberately small. That makes it easier to compare runs after changing one or two configuration values.

In [ ]:
metrics = load_json(eval_dir / "metrics.json")
display_metrics = dict(metrics)
for key in ("checkpoint_path", "embeddings_path", "projection_figure", "nearest_neighbor_figure"):
    value = display_metrics.get(key)
    if value:
        value_path = Path(value)
        if value_path.is_absolute() and REPO_ROOT in value_path.parents:
            display_metrics[key] = str(value_path.relative_to(REPO_ROOT))
        else:
            display_metrics[key] = str(value_path)
display(Markdown(format_metrics_markdown(display_metrics)))


## How To Read These Numbers

- **Linear probe accuracy** asks whether a shallow supervised layer can extract class information from the embeddings.
- **kNN accuracy** asks whether simple neighborhood structure already reflects class similarity.
- If linear probe rises but kNN stays low, the representation may contain useful information but not be well organized geometrically.
- If both improve, that is a stronger sign that the encoder is learning reusable structure.

## Configured Projection

This figure is generated by the evaluation script using `analysis.projection_method` from the config file. It should match the projection method recorded in `metrics.json` for the run you are inspecting.


In [ ]:
configured_projection_method = str(metrics.get("projection_method", "pca")).upper()
display(Markdown(f"Configured projection method: `{configured_projection_method}`"))
projection = open_saved_image(eval_dir / "projection.png")
projection


## Nearest-Neighbor Inspection

The nearest-neighbor grid is often the most intuitive qualitative check. If the left query image is consistently matched with semantically similar training images, the embedding space is doing something sensible.

In [ ]:
neighbors = open_saved_image(eval_dir / "nearest_neighbors.png")
neighbors

## Suggested Comparisons

When you rerun MAE with different settings, compare:

- `mask_ratio`
- `patch_size`
- encoder depth and width
- total pretraining epochs
- `analysis.projection_method` if you want to inspect the same run with PCA versus `t-SNE`

Try to change one factor at a time, then compare both the metrics and the qualitative artifacts. That is more informative than chasing a single scalar score.
